# ML: Forward Stockout Prediction

Predicts stockout risk with a distributed Spark `LogisticRegression` over
stock-versus-demand features and a held-out isotonic probability calibrator.

## Model contract
- Historical training snapshots are the last inventory state for each
  store/product/day.
- Labels use stockout events on future dates only, through the configured
  horizon.
- Train, probability-calibration, and test labels are partitioned by label
  availability with a three-day horizon embargo between partitions.
- Features describe the physical driver of a stockout: current stock, demand
  velocity, and stock cover over the forecast horizon (log-scaled), plus demand
  trend and day of week. Stockouts are rare, so a deep tree ensemble memorized
  the few low-stock examples (and calendar dates) and under-predicted real
  low-stock risk; this monotone model keeps low-stock positions ranked first.
- `stockout_probability` is calibrated; the raw model score is never published
  as a probability.
- Gold output preserves each position's source timestamps, records publication
  time in `generated_at`, and contains one row per store/product/horizon.
- Current positions are rolled forward to the source as-of time (the latest
  receipt): each position carries its last ledger balance (`inventory_as_of`
  is when that balance was recorded) and its demand windows end at the as-of
  time, so a slow mover's months-old snapshot is never scored as today's risk.
  Ledger rows after the as-of time (scheduled deliveries) are ignored.


In [ ]:
from datetime import timedelta
import os

import mlflow
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.regression import IsotonicRegression
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException
from pyspark.sql.window import Window


In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================


def get_env(var_name, default=None):
    return os.environ.get(var_name, default)


LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", default="ag")
GOLD_DB = get_env("GOLD_DB", default="au")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="stockout_prediction")
INVENTORY_TXN_TABLE = get_env(
    "INVENTORY_TXN_TABLE", default="fact_store_inventory_txn"
)
RECEIPT_LINES_TABLE = get_env(
    "RECEIPT_LINES_TABLE", default="fact_receipt_lines"
)
RECEIPTS_TABLE = get_env("RECEIPTS_TABLE", default="fact_receipts")
PRODUCTS_TABLE = get_env("PRODUCTS_TABLE", default="dim_products")
STOCKOUT_RISK_TABLE = get_env("STOCKOUT_RISK_TABLE", default="stockout_risk")
STOCKOUT_RISK_TABLE_NAME = (
    f"{LAKEHOUSE_NAME}.{GOLD_DB}.{STOCKOUT_RISK_TABLE}"
)
SCHEDULE_CADENCE = get_env("SCHEDULE_CADENCE", default="daily")
LABEL_COL = "stockout_label"

FORECAST_HORIZON_DAYS = int(
    get_env("FORECAST_HORIZON_DAYS", default="3")
)
LOOKBACK_DAYS = int(get_env("LOOKBACK_DAYS", default="30"))
TRAINING_HISTORY_DAYS = int(
    get_env("TRAINING_HISTORY_DAYS", default="180")
)
STOCKOUT_THRESHOLD = float(get_env("STOCKOUT_THRESHOLD", default="0"))
TRAIN_FRACTION = float(get_env("TRAIN_FRACTION", default="0.6"))
CALIBRATION_FRACTION = float(
    get_env("CALIBRATION_FRACTION", default="0.2")
)

MAX_ITER = int(get_env("MAX_ITER", default="100"))
REG_PARAM = float(get_env("REG_PARAM", default="0.0"))
# Stock cover beyond this many horizons carries no additional risk signal.
MAX_HORIZON_COVER = 999.0

if min(FORECAST_HORIZON_DAYS, LOOKBACK_DAYS, TRAINING_HISTORY_DAYS) < 1:
    raise ValueError("Forecast, feature, and training windows must be positive.")
if not 0.0 < TRAIN_FRACTION < 1.0:
    raise ValueError("TRAIN_FRACTION must be between 0 and 1.")
if not 0.0 < CALIBRATION_FRACTION < 1.0:
    raise ValueError("CALIBRATION_FRACTION must be between 0 and 1.")
if TRAIN_FRACTION + CALIBRATION_FRACTION >= 1.0:
    raise ValueError("Train and calibration fractions must leave a test window.")

print(f"Configuration: SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"Output table: {STOCKOUT_RISK_TABLE_NAME}")
print(f"Forecast horizon: {FORECAST_HORIZON_DAYS} days")
print(f"Demand feature lookback: {LOOKBACK_DAYS} days")
print(f"Training history: {TRAINING_HISTORY_DAYS} days")

# Physical ML contract used by repository validation and the pre-write gate.
ML_SOURCE_TABLES = ('fact_store_inventory_txn', 'fact_receipt_lines', 'fact_receipts', 'dim_products')
ML_OUTPUT_CONTRACTS = {'stockout_risk': [('store_id', 'long', False),
                   ('product_id', 'long', False),
                   ('risk_id', 'string', False),
                   ('inventory_as_of', 'timestamp', False),
                   ('current_inventory', 'long', False),
                   ('demand_velocity_daily', 'double', False),
                   ('days_of_inventory', 'double', False),
                   ('demand_trend', 'double', False),
                   ('stockout_probability', 'double', False),
                   ('stockout_predicted', 'int', False),
                   ('risk', 'string', False),
                   ('ranking', 'int', False),
                   ('risk_level', 'string', False),
                   ('predicted_at', 'timestamp', False),
                   ('generated_at', 'timestamp', False),
                   ('forecast_horizon_days', 'int', False),
                   ('Department', 'string', True),
                   ('Category', 'string', True),
                   ('Subcategory', 'string', True),
                   ('model_run_id', 'string', False),
                   ('schema_version', 'string', False)]}


In [ ]:
# SHARED SOURCE (generated - do not edit): retail_setup/generation/job_locks.py
# Edit utility/src/retail_setup/generation/job_locks.py, then run utility/scripts/build_notebooks.py.

"""Lease locks that keep Lakehouse writers from overlapping.

``streaming-data-load`` (03, 04), ``daily-maintenance`` (05 and the ML
refresh), ``ml-required`` (setup-11 and the ML producers) and notebook 90 run
on independent schedules and write the same Delta tables. Fabric has no
cross-pipeline mutex, so each writer claims a lease row in
``<silver>._job_locks`` before it writes:

- ``silver``: notebook 03 and notebook 90 (all Silver tables and ``_watermarks``);
- ``gold``: notebook 04 and setup-11 (the ten Gold aggregates, ``setup_run_log``);
- ``ml:<table>``: the producer of that ML output table.

Notebook 05 claims the same lock for each table it maintains, without
waiting, and skips the table when another job holds it.

A claim is a Delta ``MERGE`` that succeeds only when the row is missing,
expired, or already ours; concurrent claims conflict in Delta and one of them
loses. The holder renews its lease from a background heartbeat, so a crashed
notebook releases its locks once the lease expires (``JOB_LOCK_TTL_SECONDS``).
All times come from Spark's ``current_timestamp()``, never the driver clock.

Keep this module free of package imports: Lakehouse notebooks run it as a
cell (``scripts/build_notebooks.py`` copies it in).
"""

import threading
import time as _time
import uuid
from collections.abc import Callable, Iterable
from datetime import datetime, timedelta, timezone

JOB_LOCK_TABLE = "_job_locks"
JOB_LOCK_TTL_SECONDS = 900
JOB_LOCK_HEARTBEAT_SECONDS = 120
JOB_LOCK_POLL_SECONDS = 30
SILVER_LOCK = "silver"
GOLD_LOCK = "gold"


def ml_output_lock(table: str) -> str:
    """The lock that guards one ML output table."""
    return f"ml:{table}"


def is_concurrent_write_error(exc: BaseException) -> bool:
    """True for Delta optimistic-concurrency conflicts (another writer won)."""
    text = f"{type(exc).__name__}: {exc}"
    return "Concurrent" in text or "DELTA_CONCURRENT" in text


class DeltaJobLockStore:
    """Lease rows in a Delta table: (lock_name, owner, acquired_at, expires_at)."""

    def __init__(self, spark, table_name: str, ttl_seconds: int = JOB_LOCK_TTL_SECONDS):
        self.spark = spark
        self.table_name = table_name
        self.ttl_seconds = int(ttl_seconds)
        self.spark.sql(
            f"CREATE TABLE IF NOT EXISTS {table_name} "
            "(lock_name STRING, owner STRING, acquired_at TIMESTAMP, expires_at TIMESTAMP) "
            "USING DELTA"
        )

    @staticmethod
    def _quote(value: str) -> str:
        return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"

    def holder(self, name: str) -> str | None:
        """The owner of a live lease on ``name``, or None."""
        rows = self.spark.sql(
            f"SELECT owner FROM {self.table_name} "
            f"WHERE lock_name = {self._quote(name)} AND expires_at > current_timestamp()"
        ).collect()
        return rows[0]["owner"] if rows else None

    def try_claim(self, name: str, owner: str) -> bool:
        try:
            self.spark.sql(
                f"""
                MERGE INTO {self.table_name} AS t
                USING (
                    SELECT {self._quote(name)} AS lock_name, {self._quote(owner)} AS owner,
                           current_timestamp() AS acquired_at,
                           current_timestamp() + INTERVAL {self.ttl_seconds} SECONDS AS expires_at
                ) AS s
                ON t.lock_name = s.lock_name
                WHEN MATCHED AND (t.expires_at <= s.acquired_at OR t.owner = s.owner)
                    THEN UPDATE SET t.owner = s.owner, t.acquired_at = s.acquired_at,
                                    t.expires_at = s.expires_at
                WHEN NOT MATCHED THEN INSERT (lock_name, owner, acquired_at, expires_at)
                    VALUES (s.lock_name, s.owner, s.acquired_at, s.expires_at)
                """
            )
        except Exception as exc:
            if is_concurrent_write_error(exc):
                return False
            raise
        return self.holder(name) == owner

    def renew(self, name: str, owner: str) -> bool:
        try:
            self.spark.sql(
                f"UPDATE {self.table_name} "
                f"SET expires_at = current_timestamp() + INTERVAL {self.ttl_seconds} SECONDS "
                f"WHERE lock_name = {self._quote(name)} AND owner = {self._quote(owner)}"
            )
        except Exception as exc:
            if not is_concurrent_write_error(exc):
                raise
        return self.holder(name) == owner

    def release(self, name: str, owner: str) -> None:
        for _attempt in range(5):
            try:
                self.spark.sql(
                    f"DELETE FROM {self.table_name} "
                    f"WHERE lock_name = {self._quote(name)} AND owner = {self._quote(owner)}"
                )
                return
            except Exception as exc:
                if not is_concurrent_write_error(exc):
                    raise
                _time.sleep(1)


class MemoryJobLockStore:
    """The same lease semantics in memory, with an injectable clock (tests)."""

    def __init__(
        self,
        ttl_seconds: int = JOB_LOCK_TTL_SECONDS,
        clock: Callable[[], datetime] | None = None,
    ):
        self.ttl = timedelta(seconds=ttl_seconds)
        self.clock = clock or (lambda: datetime.now(timezone.utc).replace(tzinfo=None))
        self.rows: dict[str, tuple[str, datetime]] = {}

    def holder(self, name: str) -> str | None:
        row = self.rows.get(name)
        return row[0] if row and row[1] > self.clock() else None

    def try_claim(self, name: str, owner: str) -> bool:
        now = self.clock()
        row = self.rows.get(name)
        if row is None or row[1] <= now or row[0] == owner:
            self.rows[name] = (owner, now + self.ttl)
        return self.holder(name) == owner

    def renew(self, name: str, owner: str) -> bool:
        row = self.rows.get(name)
        if row is not None and row[0] == owner:
            self.rows[name] = (owner, self.clock() + self.ttl)
        return self.holder(name) == owner

    def release(self, name: str, owner: str) -> None:
        if self.rows.get(name, ("", None))[0] == owner:
            del self.rows[name]


def new_lock_owner(holder: str) -> str:
    """A unique owner for one notebook run, e.g. ``03-streaming-to-silver:1a2b...``."""
    return f"{holder}:{uuid.uuid4().hex[:12]}"


def acquire_job_locks(
    store,
    names: Iterable[str],
    owner: str,
    *,
    wait_seconds: float,
    poll_seconds: float = JOB_LOCK_POLL_SECONDS,
    sleep: Callable[[float], None] = _time.sleep,
    monotonic: Callable[[], float] = _time.monotonic,
) -> list[str] | None:
    """Claim every lock in ``names`` or none of them.

    Locks are claimed in sorted order, so two jobs that need several locks
    can't deadlock. Retries until ``wait_seconds`` elapse (0 = one attempt)
    and returns the claimed names, or None when another job still holds one.
    """
    ordered = sorted(set(names))
    deadline = monotonic() + max(wait_seconds, 0)
    while True:
        claimed = []
        for name in ordered:
            if not store.try_claim(name, owner):
                break
            claimed.append(name)
        if len(claimed) == len(ordered):
            return claimed
        for name in reversed(claimed):
            store.release(name, owner)
        remaining = deadline - monotonic()
        if remaining <= 0:
            return None
        sleep(min(poll_seconds, remaining))


class JobLocks:
    """A notebook's set of leases, renewed by a heartbeat thread until released."""

    def __init__(
        self,
        store,
        holder: str,
        names: Iterable[str],
        heartbeat_seconds: float = JOB_LOCK_HEARTBEAT_SECONDS,
    ):
        self.store = store
        self.owner = new_lock_owner(holder)
        self.names = sorted(set(names))
        self.heartbeat_seconds = heartbeat_seconds
        self.held: list[str] = []
        self.lost: list[str] = []
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None

    def blockers(self) -> dict[str, str | None]:
        """Current holders of the locks this job needs (for messages)."""
        return {name: self.store.holder(name) for name in self.names}

    def acquire(self, wait_seconds: float, **kwargs) -> bool:
        claimed = acquire_job_locks(
            self.store, self.names, self.owner, wait_seconds=wait_seconds, **kwargs
        )
        if claimed is None:
            return False
        self.held = claimed
        if self.heartbeat_seconds > 0:
            self._thread = threading.Thread(target=self._heartbeat, daemon=True)
            self._thread.start()
        return True

    def _heartbeat(self) -> None:
        while not self._stop.wait(self.heartbeat_seconds):
            self.renew()

    def renew(self) -> None:
        for name in self.held:
            try:
                renewed = self.store.renew(name, self.owner)
            except Exception as exc:
                print(f"Lock {name}: renewal failed ({exc}); retrying at the next heartbeat")
                continue
            if not renewed and name not in self.lost:
                self.lost.append(name)

    def assert_held(self) -> None:
        """Fail before a commit point if a lease expired and another job took it."""
        if self.lost:
            raise RuntimeError(f"Job locks lost before completion: {self.lost}")

    def release(self) -> None:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=self.heartbeat_seconds + 5)
        for name in reversed(self.held):
            self.store.release(name, self.owner)
        self.held = []

In [ ]:
# One writer per ML output: this notebook runs in more than one pipeline, and
# notebook 05 maintains the same table (see job_locks).
LOCK_WAIT_SECONDS = int(get_env("LOCK_WAIT_SECONDS", default="1800"))
JOB_LOCKS = JobLocks(
    DeltaJobLockStore(spark, f"{LAKEHOUSE_NAME}.{SILVER_DB}.{JOB_LOCK_TABLE}"),
    "12-ml-stockout-prediction",
    [ml_output_lock("stockout_risk")],
)
if not JOB_LOCKS.acquire(wait_seconds=LOCK_WAIT_SECONDS):
    raise RuntimeError(f"ML output locked by another job: {JOB_LOCKS.blockers()}")

In [ ]:
# =============================================================================
# HELPERS & MLFLOW SETUP
# =============================================================================


def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")


def read_silver(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")


def silver_exists(table_name):
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False


def resolve_table_column(frame, table_name, *candidates):
    available = {column.lower(): column for column in frame.columns}
    for candidate in candidates:
        resolved = available.get(candidate.lower())
        if resolved is not None:
            return resolved
    raise RuntimeError(
        f"Unable to resolve any of {candidates} in {table_name}. "
        f"Available columns: {frame.columns}"
    )


def chronological_split_boundaries(
    ordered_dates,
    train_fraction=0.6,
    calibration_fraction=0.2,
    embargo_days=0,
):
    """Return chronological label-availability splits with horizon embargoes."""
    dates = sorted(set(ordered_dates))
    if len(dates) < 3:
        raise ValueError(
            "Chronological train/calibration/test requires at least 3 dates."
        )
    if embargo_days < 0:
        raise ValueError("Embargo days cannot be negative.")
    usable_span_days = (dates[-1] - dates[0]).days - (2 * embargo_days)
    if usable_span_days < 3:
        raise ValueError("Insufficient label-availability span after embargoes.")

    train_days = max(1, int(usable_span_days * train_fraction))
    calibration_days = max(1, int(usable_span_days * calibration_fraction))
    train_target = dates[0] + timedelta(days=train_days)
    train_end = max(date_value for date_value in dates if date_value <= train_target)
    calibration_start = next(
        (
            date_value
            for date_value in dates
            if date_value > train_end + timedelta(days=embargo_days)
        ),
        None,
    )
    if calibration_start is None:
        raise ValueError("No calibration dates remain after the training embargo.")
    calibration_target = calibration_start + timedelta(days=calibration_days)
    calibration_end = max(
        date_value
        for date_value in dates
        if calibration_start <= date_value <= calibration_target
    )
    test_start = next(
        (
            date_value
            for date_value in dates
            if date_value > calibration_end + timedelta(days=embargo_days)
        ),
        None,
    )
    if test_start is None:
        raise ValueError("No test dates remain after the calibration embargo.")
    return {
        "train_end": train_end,
        "calibration_start": calibration_start,
        "calibration_end": calibration_end,
        "test_start": test_start,
    }


def assert_unique_keys(frame, key_columns, context):
    duplicates = (
        frame.groupBy(*key_columns)
        .count()
        .filter(F.col("count") != 1)
        .limit(1)
        .count()
    )
    if duplicates:
        raise RuntimeError(f"{context} is not unique by {key_columns}.")


def require_binary_classes(frame, label_column, context):
    if frame.select(label_column).distinct().count() != 2:
        raise RuntimeError(
            f"{context} must contain both classes for fail-closed training."
        )


def build_end_of_day_inventory(inventory_events):
    """Select final usable stock; signed ledger deficits mean zero available units."""
    end_of_day_window = Window.partitionBy(
        "store_id", "product_id", "event_date"
    ).orderBy(
        F.desc("event_ts"),
        F.desc_nulls_last("trace_id"),
        F.desc("balance"),
    )
    return (
        inventory_events.withColumn(
            "end_of_day_row", F.row_number().over(end_of_day_window)
        )
        .filter(F.col("end_of_day_row") == 1)
        .select(
            "store_id",
            "product_id",
            F.col("event_date").alias("snapshot_date"),
            F.col("event_ts").alias("inventory_as_of"),
            F.col("event_ts").alias("features_as_of"),
            F.when(
                F.col("balance").isNotNull(),
                F.greatest(F.col("balance"), F.lit(0.0)),
            ).alias("current_inventory"),
        )
    )


def roll_forward_current_positions(inventory_eod, as_of_timestamp):
    """The latest end-of-day position per store/product, carried forward to
    ``as_of_timestamp``: the balance and ``inventory_as_of`` stay as recorded,
    while ``snapshot_date`` and ``features_as_of`` move to the as-of time so
    demand windows are recomputed up to now."""
    latest_position_window = Window.partitionBy(
        "store_id", "product_id"
    ).orderBy(F.desc("snapshot_date"), F.desc("inventory_as_of"))
    return (
        inventory_eod.filter(F.col("inventory_as_of") <= F.lit(as_of_timestamp))
        .withColumn("current_row", F.row_number().over(latest_position_window))
        .filter(F.col("current_row") == 1)
        .drop("current_row")
        .withColumn("snapshot_date", F.lit(as_of_timestamp.date()).cast("date"))
        .withColumn("features_as_of", F.lit(as_of_timestamp).cast("timestamp"))
    )


def build_snapshot_features(snapshot_frame, sales, products):
    """Build trailing demand features as of each snapshot's ``features_as_of``."""
    snapshot_sales = (
        snapshot_frame.alias("snapshot")
        .join(
            sales.alias("sale"),
            on=(
                (F.col("snapshot.store_id") == F.col("sale.store_id"))
                & (
                    F.col("snapshot.product_id")
                    == F.col("sale.product_id")
                )
                & (
                    F.col("sale.sales_event_ts")
                    <= F.col("snapshot.features_as_of")
                )
                & (
                    F.col("sale.sales_event_date")
                    >= F.date_sub(
                        F.col("snapshot.snapshot_date"), LOOKBACK_DAYS - 1
                    )
                )
            ),
            how="left",
        )
        .select(
            F.col("snapshot.store_id").alias("store_id"),
            F.col("snapshot.product_id").alias("product_id"),
            F.col("snapshot.snapshot_date").alias("snapshot_date"),
            F.col("snapshot.inventory_as_of").alias("inventory_as_of"),
            F.col("snapshot.current_inventory").alias("current_inventory"),
            F.col("sale.quantity").alias("quantity"),
            F.datediff(
                F.col("snapshot.snapshot_date"),
                F.col("sale.sales_event_date"),
            ).alias("days_ago"),
        )
    )
    demand_by_snapshot = snapshot_sales.groupBy(
        "store_id",
        "product_id",
        "snapshot_date",
        "inventory_as_of",
        "current_inventory",
    ).agg(
        F.sum(
            F.when(F.col("days_ago") < 7, F.col("quantity")).otherwise(0)
        ).alias("demand_7d"),
        F.sum(
            F.when(F.col("days_ago") < 14, F.col("quantity")).otherwise(0)
        ).alias("demand_14d"),
        F.sum(
            F.when(F.col("days_ago") < 30, F.col("quantity")).otherwise(0)
        ).alias("demand_30d"),
        F.avg(
            F.when(F.col("days_ago") < 30, F.col("quantity"))
        ).alias("avg_order_size_30d"),
    )
    return (
        demand_by_snapshot.join(products, on="product_id", how="left")
        .fillna(
            0.0,
            subset=[
                "demand_7d",
                "demand_14d",
                "demand_30d",
                "avg_order_size_30d",
            ],
        )
        .withColumn("demand_velocity_daily", F.col("demand_30d") / 30.0)
        .withColumn("day_of_week", F.dayofweek("snapshot_date"))
        .withColumn("day_of_month", F.dayofmonth("snapshot_date"))
        .withColumn("week_of_year", F.weekofyear("snapshot_date"))
        .withColumn(
            "is_weekend",
            F.when(F.dayofweek("snapshot_date").isin(1, 7), 1).otherwise(0),
        )
        .withColumn(
            "days_of_inventory",
            F.when(
                F.col("demand_velocity_daily") > 0,
                F.col("current_inventory") / F.col("demand_velocity_daily"),
            ).otherwise(999.0),
        )
        .withColumn(
            "demand_trend",
            F.when(
                F.col("demand_30d") > 0,
                (F.col("demand_7d") / 7.0)
                / (F.col("demand_30d") / 30.0),
            ).otherwise(1.0),
        )
        .withColumn(
            "inventory_ratio",
            F.when(
                F.col("demand_30d") > 0,
                F.col("current_inventory") / F.col("demand_30d"),
            ).otherwise(1.0),
        )
    )


def attach_future_stockout_labels(snapshot_features, stockout_events):
    """Label stockouts on dates strictly after each end-of-day snapshot."""
    future_hits = (
        snapshot_features.select(
            "store_id", "product_id", "snapshot_date", "inventory_as_of"
        )
        .alias("snapshot")
        .join(
            stockout_events.alias("stockout"),
            on=(
                (F.col("snapshot.store_id") == F.col("stockout.store_id"))
                & (
                    F.col("snapshot.product_id")
                    == F.col("stockout.product_id")
                )
                & (
                    F.col("stockout.stockout_date")
                    > F.col("snapshot.snapshot_date")
                )
                & (
                    F.col("stockout.stockout_date")
                    <= F.date_add(
                        F.col("snapshot.snapshot_date"),
                        FORECAST_HORIZON_DAYS,
                    )
                )
            ),
            how="left",
        )
        .select(
            F.col("snapshot.store_id").alias("store_id"),
            F.col("snapshot.product_id").alias("product_id"),
            F.col("snapshot.snapshot_date").alias("snapshot_date"),
            F.col("snapshot.inventory_as_of").alias("inventory_as_of"),
            F.col("stockout.stockout_date").alias("stockout_date"),
        )
        .groupBy(
            "store_id", "product_id", "snapshot_date", "inventory_as_of"
        )
        .agg(F.count("stockout_date").alias("future_stockout_count"))
        .withColumn(
            LABEL_COL,
            (F.col("future_stockout_count") > 0).cast("double"),
        )
        .drop("future_stockout_count")
    )
    return snapshot_features.join(
        future_hits,
        on=["store_id", "product_id", "snapshot_date", "inventory_as_of"],
        how="inner",
    ).withColumn(
        "label_available_date",
        F.date_add(F.col("snapshot_date"), FORECAST_HORIZON_DAYS),
    )


def raw_stockout_scores(model, frame):
    return model.transform(frame).withColumn(
        "raw_stockout_score",
        vector_to_array("probability")[1],
    )


def score_with_calibration(model, calibrator, calibration_assembler, frame):
    raw_scores = raw_stockout_scores(model, frame)
    calibrated = calibrator.transform(
        calibration_assembler.transform(raw_scores)
    ).withColumn(
        "stockout_probability",
        F.greatest(
            F.lit(0.0),
            F.least(F.lit(1.0), F.col("calibrated_probability")),
        ),
    )
    invalid_count = calibrated.filter(
        F.col("stockout_probability").isNull()
        | (F.col("stockout_probability") < 0.0)
        | (F.col("stockout_probability") > 1.0)
    ).limit(1).count()
    if invalid_count:
        raise RuntimeError("Calibrated stockout probabilities are invalid.")
    return calibrated.withColumn(
        "stockout_predicted",
        (F.col("stockout_probability") >= 0.5).cast("double"),
    )


ensure_database(GOLD_DB)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment '{EXPERIMENT_NAME}' ready")

def _normalize_ml_type(data_type):
    value = data_type.replace("bigint", "long").replace("integer", "int")
    return value


def validate_ml_output(frame, table_name):
    contract = ML_OUTPUT_CONTRACTS[table_name]
    expected = tuple((name, data_type) for name, data_type, _ in contract)
    actual = tuple(
        (field.name, _normalize_ml_type(field.dataType.simpleString()))
        for field in frame.schema.fields
    )
    if actual != expected:
        raise RuntimeError(
            f"ML output schema mismatch for {table_name}: expected={expected}, actual={actual}"
        )
    non_nullable = tuple(name for name, _, nullable in contract if not nullable)
    null_flags = frame.agg(
        *(F.max(F.col(name).isNull().cast("int")).alias(name) for name in non_nullable)
    ).first().asDict()
    null_columns = [name for name in non_nullable if null_flags.get(name)]
    if null_columns:
        raise RuntimeError(
            f"ML output {table_name} has nulls in required columns: {null_columns}"
        )
    return frame



## Step 1: Calculate Current Inventory Position and Historical Demand


In [ ]:
required_tables = [
    INVENTORY_TXN_TABLE,
    RECEIPT_LINES_TABLE,
    RECEIPTS_TABLE,
    PRODUCTS_TABLE,
]
for table_name in required_tables:
    if not silver_exists(table_name):
        raise RuntimeError(f"{table_name} table not found in Silver layer")

print("=" * 80)
print("EXTRACTING END-OF-DAY INVENTORY AND DEMAND DATA")
print("=" * 80)

inventory_history = (
    read_silver(INVENTORY_TXN_TABLE)
    .select(
        "store_id",
        "product_id",
        F.col("event_ts").cast("timestamp").alias("event_ts"),
        F.col("trace_id").cast("string").alias("trace_id"),
        F.col("balance").cast("double").alias("balance"),
    )
    .filter(
        F.col("store_id").isNotNull()
        & F.col("product_id").isNotNull()
        & F.col("event_ts").isNotNull()
    )
    .withColumn("event_date", F.to_date("event_ts"))
)
null_balance_count = (
    inventory_history.filter(F.col("balance").isNull()).limit(1).count()
)
if null_balance_count:
    raise RuntimeError(
        f"{INVENTORY_TXN_TABLE} contains null balances; "
        "repair the inventory history instead of treating null as zero."
    )
inventory_history = (
    inventory_history
    .withColumn(
        "is_stockout",
        (F.col("balance") <= STOCKOUT_THRESHOLD).cast("int"),
    )
)
# One source as-of for every required ML output: the latest Silver receipt.
source_anchor = read_silver(RECEIPTS_TABLE).agg(
    F.max(F.col("event_ts").cast("timestamp")).alias("source_as_of_timestamp")
).first()["source_as_of_timestamp"]
if source_anchor is None:
    raise RuntimeError(f"No receipts found in {RECEIPTS_TABLE}.")
source_as_of_timestamp = source_anchor
source_as_of_date = source_as_of_timestamp.date()
# Ledger rows after the as-of time (scheduled deliveries) are not yet true.
inventory_history = inventory_history.filter(
    F.col("event_ts") <= F.lit(source_as_of_timestamp)
).cache()
if inventory_history.limit(1).count() == 0:
    raise RuntimeError(f"No inventory history found in {INVENTORY_TXN_TABLE}.")
training_start_date = source_as_of_date - timedelta(
    days=TRAINING_HISTORY_DAYS
)
label_cutoff_date = source_as_of_date - timedelta(
    days=FORECAST_HORIZON_DAYS
)
sales_history_start_date = training_start_date - timedelta(days=LOOKBACK_DAYS)

inventory_eod = build_end_of_day_inventory(inventory_history).cache()
assert_unique_keys(
    inventory_eod,
    ["store_id", "product_id", "snapshot_date"],
    "End-of-day inventory",
)

current_inventory_snapshots = roll_forward_current_positions(
    inventory_eod, source_as_of_timestamp
)
historical_snapshots = inventory_eod.filter(
    (F.col("snapshot_date") >= F.lit(training_start_date))
    & (F.col("snapshot_date") <= F.lit(label_cutoff_date))
)
stockout_events = (
    inventory_history.filter(F.col("is_stockout") == 1)
    .select(
        "store_id",
        "product_id",
        F.col("event_date").alias("stockout_date"),
    )
    .distinct()
)

sales_df = (
    read_silver(RECEIPT_LINES_TABLE)
    .select(
        "receipt_id_ext",
        "product_id",
        F.col("quantity").cast("double").alias("quantity"),
    )
    .alias("line")
    .join(
        read_silver(RECEIPTS_TABLE)
        .select(
            "receipt_id_ext",
            "store_id",
            "receipt_type",
            F.col("event_ts").cast("timestamp").alias("sales_event_ts"),
        )
        .alias("receipt"),
        on="receipt_id_ext",
        how="inner",
    )
    .select(
        F.col("receipt.store_id").alias("store_id"),
        F.col("line.product_id").alias("product_id"),
        "quantity",
        "receipt_type",
        "sales_event_ts",
        F.to_date("sales_event_ts").alias("sales_event_date"),
    )
    .filter(
        (F.col("receipt_type") == "SALE")
        & (F.col("quantity") > 0)
        & (F.col("sales_event_date") >= F.lit(sales_history_start_date))
        & (F.col("sales_event_ts") <= F.lit(source_as_of_timestamp))
    )
    .cache()
)

print(f"Source as-of (latest receipt): {source_as_of_timestamp}")
print(f"End-of-day inventory rows: {inventory_eod.count()}")
print(f"Historical training snapshots: {historical_snapshots.count()}")
print(f"Current inventory positions: {current_inventory_snapshots.count()}")


## Step 2: Feature Engineering


In [ ]:
print("=" * 80)
print("FEATURE ENGINEERING")
print("=" * 80)

products_source = read_silver(PRODUCTS_TABLE)
product_id_column = resolve_table_column(
    products_source, PRODUCTS_TABLE, "product_id", "id"
)
department_column = resolve_table_column(
    products_source, PRODUCTS_TABLE, "department"
)
category_column = resolve_table_column(
    products_source, PRODUCTS_TABLE, "product_category", "category"
)
subcategory_column = resolve_table_column(
    products_source,
    PRODUCTS_TABLE,
    "product_subcategory",
    "subcategory",
)
products_df = (
    products_source.select(
        F.col(product_id_column).cast("long").alias("product_id"),
        F.col(department_column).alias("Department"),
        F.col(category_column).alias("Category"),
        F.col(subcategory_column).alias("Subcategory"),
    )
    .groupBy("product_id")
    .agg(
        F.min("Department").alias("Department"),
        F.min("Category").alias("Category"),
        F.min("Subcategory").alias("Subcategory"),
    )
)

features_df = build_snapshot_features(
    current_inventory_snapshots,
    sales_df,
    products_df,
).cache()
historical_features_df = build_snapshot_features(
    historical_snapshots,
    sales_df,
    products_df,
).cache()

assert_unique_keys(
    features_df,
    ["store_id", "product_id"],
    "Current stockout features",
)
assert_unique_keys(
    historical_features_df,
    ["store_id", "product_id", "snapshot_date"],
    "Historical stockout features",
)

print(f"Current feature rows: {features_df.count()}")
print(f"Historical end-of-day feature rows: {historical_features_df.count()}")
features_df.select(
    "store_id",
    "product_id",
    "inventory_as_of",
    "current_inventory",
    "demand_velocity_daily",
    "days_of_inventory",
    "demand_trend",
).show(10)


## Step 3: Create Training Labels (Historical Stockouts)


In [ ]:
print("=" * 80)
print("CREATING FUTURE-ONLY TRAINING LABELS")
print("=" * 80)

training_data = attach_future_stockout_labels(
    historical_features_df,
    stockout_events,
).cache()
assert_unique_keys(
    training_data,
    ["store_id", "product_id", "snapshot_date"],
    "Labeled stockout snapshots",
)

ordered_label_available_dates = [
    row["label_available_date"]
    for row in training_data.select("label_available_date")
    .distinct()
    .orderBy("label_available_date")
    .collect()
]
split_dates = chronological_split_boundaries(
    ordered_label_available_dates,
    TRAIN_FRACTION,
    CALIBRATION_FRACTION,
    FORECAST_HORIZON_DAYS,
)
train_label_available_end = split_dates["train_end"]
calibration_label_available_start = split_dates["calibration_start"]
calibration_label_available_end = split_dates["calibration_end"]
test_label_available_start = split_dates["test_start"]

print(f"Training records at end-of-day grain: {training_data.count()}")
training_data.groupBy(LABEL_COL).count().orderBy(LABEL_COL).show()
print(f"Train labels available through: {train_label_available_end}")
print(f"Calibration labels start after embargo: {calibration_label_available_start}")
print(f"Calibration labels available through: {calibration_label_available_end}")
print(f"Test labels available from: {test_label_available_start}")


## Step 4: Train LogisticRegression Model (Distributed)

Uses Spark ML `LogisticRegression` — training runs distributed across Spark
executors.  All data stays in Spark DataFrames (no `toPandas()`).


In [ ]:
print("=" * 80)
print("TRAINING AND CALIBRATING LOGISTIC REGRESSION")
print("=" * 80)

# Calendar dates (day of month, week of year) are deliberately excluded: with
# about a year of history they identify individual dates, not a repeatable
# stockout pattern.
feature_cols = [
    "log_inventory",
    "log_velocity",
    "log_horizon_cover",
    "demand_trend",
    "day_of_week",
    "is_weekend",
]


def prepare_model_frame(frame):
    expected_horizon_demand = (
        F.col("demand_velocity_daily").cast("double") * F.lit(float(FORECAST_HORIZON_DAYS))
    )
    prepared = (
        frame.withColumn(
            "log_inventory",
            F.log1p(F.greatest(F.col("current_inventory").cast("double"), F.lit(0.0))),
        )
        .withColumn(
            "log_velocity",
            F.log1p(F.greatest(F.col("demand_velocity_daily").cast("double"), F.lit(0.0))),
        )
        .withColumn(
            "log_horizon_cover",
            F.log1p(
                F.when(
                    expected_horizon_demand > 0,
                    F.least(
                        F.col("current_inventory").cast("double") / expected_horizon_demand,
                        F.lit(MAX_HORIZON_COVER),
                    ),
                ).otherwise(F.lit(MAX_HORIZON_COVER))
            ),
        )
    )
    for column_name in feature_cols:
        prepared = prepared.withColumn(
            column_name,
            F.col(column_name).cast("double"),
        )
    return prepared.na.fill(0.0, subset=feature_cols)


assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="error",
)
assembled_history = assembler.transform(prepare_model_frame(training_data))
df_train = assembled_history.filter(
    F.col("label_available_date") <= F.lit(train_label_available_end)
)
df_calibration = assembled_history.filter(
    (F.col("label_available_date") >= F.lit(calibration_label_available_start))
    & (
        F.col("label_available_date")
        <= F.lit(calibration_label_available_end)
    )
)
df_test = assembled_history.filter(
    F.col("label_available_date") >= F.lit(test_label_available_start)
)

for split_name, split_frame in [
    ("Training split", df_train),
    ("Calibration split", df_calibration),
    ("Test split", df_test),
]:
    if split_frame.count() == 0:
        raise RuntimeError(f"{split_name} is empty.")
    require_binary_classes(split_frame, LABEL_COL, split_name)

train_count = df_train.count()
calibration_count = df_calibration.count()
test_count = df_test.count()
print(
    f"Chronological rows -- train: {train_count}, "
    f"calibration: {calibration_count}, test: {test_count}"
)

classifier = LogisticRegression(
    featuresCol="features",
    labelCol=LABEL_COL,
    maxIter=MAX_ITER,
    regParam=REG_PARAM,
    standardization=True,
)

with mlflow.start_run(
    run_name=f"logistic_stockout_{source_as_of_timestamp.strftime('%Y%m%d_%H%M')}"
) as run:
    model = classifier.fit(df_train.select("features", LABEL_COL))

    raw_calibration = raw_stockout_scores(model, df_calibration).cache()
    if raw_calibration.select("raw_stockout_score").distinct().count() < 2:
        raise RuntimeError(
            "Stockout calibration has fewer than two distinct model scores."
        )
    calibration_assembler = VectorAssembler(
        inputCols=["raw_stockout_score"],
        outputCol="calibration_features",
        handleInvalid="error",
    )
    calibrator = IsotonicRegression(
        featuresCol="calibration_features",
        labelCol=LABEL_COL,
        predictionCol="calibrated_probability",
        isotonic=True,
    ).fit(calibration_assembler.transform(raw_calibration))

    df_predictions = score_with_calibration(
        model,
        calibrator,
        calibration_assembler,
        df_test,
    )
    binary_evaluator = BinaryClassificationEvaluator(
        rawPredictionCol="stockout_probability",
        labelCol=LABEL_COL,
        metricName="areaUnderROC",
    )
    auc_roc = float(binary_evaluator.evaluate(df_predictions))
    multiclass_evaluator = MulticlassClassificationEvaluator(
        predictionCol="stockout_predicted",
        labelCol=LABEL_COL,
    )
    precision = float(
        multiclass_evaluator.evaluate(
            df_predictions,
            {multiclass_evaluator.metricName: "weightedPrecision"},
        )
    )
    recall = float(
        multiclass_evaluator.evaluate(
            df_predictions,
            {multiclass_evaluator.metricName: "weightedRecall"},
        )
    )
    f1 = float(
        multiclass_evaluator.evaluate(
            df_predictions,
            {multiclass_evaluator.metricName: "f1"},
        )
    )
    brier_score = float(
        df_predictions.agg(
            F.avg(
                F.pow(
                    F.col("stockout_probability") - F.col(LABEL_COL),
                    2,
                )
            ).alias("brier_score")
        ).first()["brier_score"]
    )

    mlflow.log_params({
        "algorithm": "logistic_regression_isotonic_calibration",
        "forecast_horizon_days": FORECAST_HORIZON_DAYS,
        "lookback_days": LOOKBACK_DAYS,
        "training_history_days": TRAINING_HISTORY_DAYS,
        "max_iter": MAX_ITER,
        "reg_param": REG_PARAM,
        "feature_count": len(feature_cols),
        "train_rows": train_count,
        "calibration_rows": calibration_count,
        "test_rows": test_count,
        "source_as_of": source_as_of_timestamp.isoformat(),
    })
    mlflow.log_metrics({
        "calibrated_auc_roc": round(auc_roc, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "brier_score": round(brier_score, 6),
    })
    mlflow.spark.log_model(model, "logistic_stockout_model")
    mlflow.spark.log_model(calibrator, "isotonic_stockout_calibrator")

    print(f"MLflow run: {run.info.run_id}")
    print(f"Calibrated test AUC-ROC: {auc_roc:.4f}")
    print(f"Precision: {precision:.4f}; Recall: {recall:.4f}; F1: {f1:.4f}")
    print(f"Brier score: {brier_score:.6f}")


## Step 5: Generate Predictions for Current Inventory

Score the full current-inventory feature set using the trained model.
All prediction stays in Spark — no `toPandas()`.


In [ ]:
print("=" * 80)
print("GENERATING CURRENT CALIBRATED STOCKOUT RISK")
print("=" * 80)

current_assembled = assembler.transform(prepare_model_frame(features_df))
df_scored = score_with_calibration(
    model,
    calibrator,
    calibration_assembler,
    current_assembled,
)

df_scored = (
    df_scored.withColumn(
        "risk_level",
        F.when(F.col("stockout_probability") >= 0.7, "HIGH")
        .when(F.col("stockout_probability") >= 0.4, "MEDIUM")
        .otherwise("LOW"),
    )
    .withColumn("risk", F.col("risk_level"))
    .withColumn(
        "predicted_at",
        F.lit(source_as_of_timestamp).cast("timestamp"),
    )
    .withColumn(
        "forecast_horizon_days",
        F.lit(FORECAST_HORIZON_DAYS).cast("int"),
    )
)

risk_rank_window = Window.orderBy(
    F.desc("stockout_probability"),
    F.asc("store_id"),
    F.asc("product_id"),
)
df_scored = df_scored.withColumn(
    "ranking", F.row_number().over(risk_rank_window)
)
assert_unique_keys(
    df_scored,
    ["store_id", "product_id", "forecast_horizon_days"],
    "Current stockout scores",
)

print(f"Predictions generated: {df_scored.count()}")
df_scored.groupBy("risk_level").count().orderBy("risk_level").show()
df_scored.select(
    "store_id",
    "product_id",
    "inventory_as_of",
    "current_inventory",
    "demand_velocity_daily",
    "days_of_inventory",
    "stockout_probability",
    "risk_level",
).orderBy(F.desc("stockout_probability")).show(20)


## Step 6: Save to Gold Layer


In [ ]:
print("=" * 80)
print("SAVING ONE CURRENT ROW PER STORE/PRODUCT/HORIZON")
print("=" * 80)

df_output = (
    df_scored.select(
        F.col("store_id").cast("long"),
        F.col("product_id").cast("long"),
        # Stable entity id for the ontology; ranking changes on every run.
        F.concat_ws(
            ":",
            *(F.col(name).cast("string")
              for name in ("store_id", "product_id", "forecast_horizon_days")),
        ).alias("risk_id"),
        F.col("inventory_as_of").cast("timestamp"),
        F.col("current_inventory").cast("long"),
        F.col("demand_velocity_daily").cast("double"),
        F.col("days_of_inventory").cast("double"),
        F.col("demand_trend").cast("double"),
        F.col("stockout_probability").cast("double"),
        F.col("stockout_predicted").cast("int"),
        F.col("risk").cast("string"),
        F.col("ranking").cast("int"),
        F.col("risk_level").cast("string"),
        F.col("predicted_at").cast("timestamp"),
        F.current_timestamp().alias("generated_at"),
        F.col("forecast_horizon_days").cast("int"),
        F.col("Department").cast("string"),
        F.col("Category").cast("string"),
        F.col("Subcategory").cast("string"),
        F.lit(run.info.run_id).cast("string").alias("model_run_id"),
        F.lit("1.0").cast("string").alias("schema_version"),
    )
    .orderBy("store_id", "product_id", "forecast_horizon_days")
)
assert_unique_keys(
    df_output,
    ["store_id", "product_id", "forecast_horizon_days"],
    "Stockout Gold output",
)

df_output = validate_ml_output(df_output, "stockout_risk")
df_output.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(
    STOCKOUT_RISK_TABLE_NAME
)
row_count = spark.table(STOCKOUT_RISK_TABLE_NAME).count()
if row_count != current_inventory_snapshots.count():
    raise RuntimeError(
        "Published stockout output does not contain exactly one row per "
        "current store/product/horizon."
    )
print(f"{STOCKOUT_RISK_TABLE_NAME}: {row_count} rows")


In [ ]:
print("=" * 80)
print("STOCKOUT PREDICTION COMPLETE")
print("=" * 80)
print(f"Output table: {STOCKOUT_RISK_TABLE_NAME}")
print(f"Source as-of: {source_as_of_timestamp}")
print(f"Forecast horizon: {FORECAST_HORIZON_DAYS} days")
print("Training grain: end-of-day store/product")
print("Published probabilities: isotonic calibrated")
print("Published inventory timestamp: inventory_as_of")
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(
    f"Schedule this notebook on a {SCHEDULE_CADENCE} cadence for fresh "
    "predictions."
)


In [ ]:
# Release the ML output lock (see job_locks).
JOB_LOCKS.assert_held()
JOB_LOCKS.release()